# Работа с ресурсами

Выполнил студент Андрющенко Ксения

## Порядок сдачи домашнего

Под каждое домашнее вы создаете отдельную ветку куда вносите все изменения в рамках домашнего. Как только домашнее готово - создаете пулл реквест (обратите внимание что в пулл реквесте должны быть отражены все изменения в рамках домашнего). Ревьювера назначаете из таблицы - https://docs.google.com/spreadsheets/d/1vK6IgEqaqXniUJAQOOspiL_tx3EYTSXW1cUrMHAZFr8/edit?gid=0#gid=0
Перед сдачей проверьте код, напишите тесты. Не забудьте про PEP8, например, с помощью flake8. Задание нужно делать в jupyter notebook.

**Дедлайн - 14 ноября 10:00**

# Менеджер контекста для смены директории (cd)

Напишите класс менеджера контекста ChangeDir, который временно меняет текущую рабочую директорию на заданную. После выхода из контекста рабочая директория должна вернуться к предыдущей.

**Условия:**
1.	При входе в блок with менеджер контекста должен изменить текущую директорию на указанную.
2.	При выходе из блока with менеджер контекста должен вернуть рабочую директорию на исходное значение.
3.	Обработайте ситуацию, когда указанный путь не существует, с выводом сообщения об ошибке.

**Пример:**

```python
import os

print("Начальная директория:", os.getcwd())

with ChangeDir("/path/to/new/directory"):
    print("Внутри менеджера:", os.getcwd())

print("После выхода:", os.getcwd())
```

* Метод __enter__ класса resourses. Если
конструкция with включает в себя слово as, то
возвращаемое методом __enter__ значение записывается
в переменную.
* Вызывается метод __exit__ класса resourses. Причём в
метод __exit__ передаются три параметра - тип
исключения, исключение и traceback.

*Функция `chdir()` модуля os изменяет текущий рабочий каталог.*
*`isdir()` модуля os.path возвращает True если путь path существует и является каталогом*

In [48]:
import os
import unittest
import tempfile
import sys
import io
import time
import random

In [49]:
class ChabgeDir:
    def __init__(self, dnmae):
        # заданная директория
        self.dname = os.path.join(dnmae)
    def __enter__(self):
        # ntreofz lbhtrnjhbz
        self.current_dir = os.getcwd()
        # изменяем рабочий каталог
        # ghjdthztv rjhhtrnyjcnm genb
        if os.path.isdir(self.dname):
            os.chdir(self.dname)
        else:
            raise NameError('he object does not exist or is not a directory')
    def __exit__(self, exp_type, exp_value, traceback):
        # обратно изменяем рабочий каталог
        os.chdir(self.current_dir)

Тестировнние

In [65]:
class TestChangeDir(unittest.TestCase):
    def setUp(self):
        # временные каталоги для тестов
        self.original_dir = os.getcwd()
        self.temp_dir = tempfile.TemporaryDirectory()

    def tearDown(self):
        # удаление тестовых каталогов
        self.temp_dir.cleanup()

    def test_change(self):
        # переход в существующую директорию
        with ChabgeDir(self.temp_dir.name):
            self.assertEqual(os.getcwd(), self.temp_dir.name)
        # после выхода из контекста, директория должна вернуться на исходную
        self.assertEqual(os.getcwd(), self.original_dir)

    def test_change_fail(self):
        # переход в несуществующую директорию
        non_existing_dir = os.path.join(self.temp_dir.name, 'non_existing_dir')
        with self.assertRaises(NameError):
            with ChabgeDir(non_existing_dir):
                pass  # исключение

    def test_change_back(self):
        # директория возвращается к исходной после использования
        with ChabgeDir(self.temp_dir.name):
            self.assertEqual(os.getcwd(), self.temp_dir.name)
        self.assertEqual(os.getcwd(), self.original_dir)

In [66]:
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestChangeDir))

...
----------------------------------------------------------------------
Ran 3 tests in 0.009s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

# Перенаправления вывода в файл

Напишите класс менеджера контекста RedirectOutput, который временно перенаправляет стандартный поток вывода stdout в указанный файл. После выхода из контекста вывод должен возвращаться в стандартный поток.

**Условия:**

1.	При входе в блок with менеджер контекста должен перенаправить вывод print в файл, указанный при создании объекта.
2.	При выходе из блока with вывод должен возвращаться в стандартный поток.
3.	Если файл уже существует, вывод должен дописываться к нему, а не перезаписывать его.

**Пример:**
```python
print("Это стандартный вывод")  # Должно выводиться в консоль

with RedirectOutput("output.txt"):
    print("Это вывод в файл")   # Должно записываться в файл "output.txt"

print("Снова стандартный вывод")  # Должно выводиться в консоль
```


In [52]:
class RedirectOutput:
    def __init__(self, fname):
        # для перенаправления ввода
        self.fname = fname
        self.original_stdout = None

    def __enter__(self):
        # текущий стандартный вывод
        self.original_stdout = sys.stdout
        self.file = open(self.fname, 'a')
        # стандартный вывод на новый файл
        sys.stdout = self.file

    def __exit__(self, exp_type, exp_value, traceback):
        # возвращаем стандартный вывод обратно
        sys.stdout = self.original_stdout
        # закрываем файл если открыт
        self.file.close()

In [53]:
print("Это стандартный вывод")  # Должно выводиться в консоль

with RedirectOutput("output.txt"):
    print("Это вывод в файл")   # Должно записываться в файл "output.txt"

print("Снова стандартный вывод")  # Должно выводиться в консоль

Это стандартный вывод
Снова стандартный вывод


Тестирование

In [69]:
class TestOutputRedirector(unittest.TestCase):
    def test_basic_redirection(self):
        # оригинальный stdout
        original_stdout = sys.stdout
        # Временный файл для проверки
        test_file = "test_output.txt"
        if os.path.exists(test_file):
            os.remove(test_file)
        # Перехват стандартного вывода
        sys.stdout = io.StringIO()
        print("Консоль")
        # перенаправляем вывод в файл
        with RedirectOutput(test_file):
            print("Файл")
        print("Консоль")
        # Проверяем вывод в консоль
        output_console = sys.stdout.getvalue()
        self.assertEqual(output_console, "Консоль\nКонсоль\n")
        sys.stdout = original_stdout
        # проверка содержимого файла
        with open(test_file, 'r') as f:
            file_content = f.read().strip()
        self.assertEqual(file_content, "Файл")
        # удаляем файл после проверки
        os.remove(test_file)

In [70]:
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestOutputRedirector))

.
----------------------------------------------------------------------
Ran 1 test in 0.006s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

# Замер времени выполнения кода

Напишите класс менеджера контекста Timer, который замеряет время выполнения кода внутри блока with. Менеджер должен выводить время выполнения в консоль по завершении блока. Для замера времени используйте модуль time.

**Условия:**
1. При входе в блок with менеджер контекста должен начинать отсчёт времени.
2. При выходе из блока with менеджер должен выводить в консоль время выполнения кода внутри блока в формате "Время выполнения: X.XXX секунд".
3. Опционально: добавить возможность передавать имя таймера при инициализации, чтобы можно было различать результаты замеров, если их несколько.

**Пример:**
```python
import time

with Timer("Задача 1"):
    time.sleep(1)  # Симуляция работы кода
[Задача 1] Время выполнения: 1.001 секунд
    
with Timer("Задача 2"):
    for i in range(1000000):
        pass
[Задача 2] Время выполнения: 0.034 секунд
```

In [55]:
class Timer:
    def __init__(self, task):
        self.task = task

    def __enter__(self):
        self.start = time.time()

    def __exit__(self, exp_type, exp_value, traceback):
        self.end = time.time()
        print(f"[{self.task}] Время выполнения: {round(self.end - self.start, 3)} секунды")

Тестирование

In [56]:
with Timer("Задача 1"):
    time.sleep(1)  # Симуляция работы кода

with Timer("Задача 2"):
    for i in range(100000):
        pass

[Задача 1] Время выполнения: 1.0 секунды
[Задача 2] Время выполнения: 0.01 секунды


# Поглощение исключения

Напишите класс менеджера контекста SuppressExceptions, который подавляет указанные исключения внутри блока with, не прерывая выполнение программы. Если в блоке возникает исключение, которое не входит в список подавляемых, оно должно быть выброшено обычным образом.

**Условия:**
1.	При инициализации менеджера контекста нужно передавать типы исключений, которые будут подавляться.
2.	Если в блоке with возникает исключение из списка подавляемых, оно должно игнорироваться.
3.	Если возникает исключение, не входящее в список, оно должно быть выброшено.
4.	Опционально: после подавления исключения вывести сообщение о том, какое исключение было подавлено.


**Пример:**
```python
with SuppressExceptions(ZeroDivisionError, ValueError):
    print(1 / 0)  # Это исключение будет подавлено

with SuppressExceptions(TypeError):
    print(1 + "2")  # Это исключение будет подавлено

with SuppressExceptions(IndexError):
    print([1, 2, 3][5])  # Это исключение будет подавлено

print("Программа продолжает работать после блока with")
```

In [57]:
class SuppressExceptions:
    def __init__(self, *exp):
        self.exp = list(exp)

    def __enter__(self):
        return self

    def __exit__(self, exp_type, exp_value, traceback):
        if exp_type in self.exp:
            print("исключение было подавлено")
            return True
        else:
            return False

In [58]:
with SuppressExceptions(ZeroDivisionError, ValueError):
    print(1 / 0)  # Это исключение будет подавлено

with SuppressExceptions(TypeError):
    print(1 + "2")  # Это исключение будет подавлено

with SuppressExceptions(IndexError):
    print([1, 2, 3][5])  # Это исключение будет подавлено

print("Программа продолжает работать после блока with")

исключение было подавлено
исключение было подавлено
исключение было подавлено
Программа продолжает работать после блока with


# Создание временного файла
Напишите класс менеджера контекста TemporaryFile, который создаёт временный файл при входе в контекст и автоматически удаляет его при выходе. Менеджер должен позволять записывать и читать данные из файла в течение его существования в контексте.

**Условия:**
1.	При входе в блок with менеджер должен создавать временный файл и возвращать его объект для записи и чтения.
2.	При выходе из блока with временный файл должен автоматически удаляться.
3.	Имя файла должно быть уникальным и генерироваться автоматически.

**Пример**
```python
with TemporaryFile() as temp_file:
    temp_file.write(b"Временные данные\n")  # Записываем данные
    temp_file.seek(0)  # Возвращаемся в начало файла
    print(temp_file.read())  # Читаем данные из временного файла

print("Файл автоматически удалён")
```

In [60]:
class TemporaryFile():
    #def __init__(self):
    #    print("TemporaryFile")
    def __enter__(self):
        self.temp_f = tempfile.NamedTemporaryFile()
        return self

    def write(self, data):
        self.temp_f.write(bytes(data, 'utf-8'))
    
    def seek(self, search):
        self.temp_f.seek(search)

    def read(self):
        return self.temp_f.read().decode("utf-8")

    def __exit__(self, exp_type, exp_value, exp_traceback):
        self.temp_f.close()
        return True

In [61]:
with TemporaryFile() as temp_file:
    temp_file.write("Временные данные\n")  # Записываем данные
    temp_file.seek(0)  # Возвращаемся в начало файла
    print(temp_file.read())  # Читаем данные из временного файла

print("Файл автоматически удалён")

Временные данные

Файл автоматически удалён
